# 🅿️ Lisbon Parking Analytics
## Phase 1 — Exploratory Data Analysis

**Goal:** Ingest and explore public datasets on Lisbon parking to understand the data structure before modelling in SQL.

**Datasets used:**
| # | Name | Source | What it contains |
|---|---|---|---|
| 1 | On-street parking spots (EMEL) | Lisboa Dados Abertos | Location, zone, tariff, type of each spot |
| 2 | Regulated parking areas | Lisboa Dados Abertos | Polygon zones with tariff and schedule info |
| 3 | Parking spot zones (ZEDL/ZAAC) | Lisboa Dados Abertos | Zone boundaries (residents vs public) |
| 4 | EMEL financial context | Manually entered | Revenue figures from 2024 annual report |

---

## 0. Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import requests
import json
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Paths
RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

print('✅ Libraries loaded')

---
## 1. Dataset 1 — On-Street Parking Spots (EMEL)

**Source:** Lisboa Dados Abertos  
**URL:** https://dadosabertos.cm-lisboa.pt/dataset/emel-lugares-de-estacionamento-via-publica  
**What it contains:** Location and attributes of every on-street parking spot managed by EMEL — zone, tariff tier, spot type, payment method, schedule.

In [ ]:
# ── Fetch from Lisboa Dados Abertos CKAN API ──────────────────────────────────
SPOTS_API = (
    "https://dadosabertos.cm-lisboa.pt/api/3/action/datastore_search"
    "?resource_id=1a4ddee5-e5ef-497c-8e39-b89ddddc5e93"
    "&limit=32000"
)

resp = requests.get(SPOTS_API, timeout=30)
resp.raise_for_status()
data = resp.json()

df_spots = pd.DataFrame(data['result']['records'])
print(f'Records fetched: {len(df_spots):,}')
print(f'Columns: {list(df_spots.columns)}')

In [ ]:
# ── If the API is unavailable, load from local file (see README) ──────────────
# Uncomment the lines below and download the CSV manually from the URL above
# df_spots = pd.read_csv(RAW_DIR + 'emel_spots.csv')

df_spots.head(3)

In [ ]:
# ── Basic shape & types ───────────────────────────────────────────────────────
print('Shape:', df_spots.shape)
print()
df_spots.dtypes

In [ ]:
# ── Nulls ─────────────────────────────────────────────────────────────────────
null_pct = (df_spots.isnull().sum() / len(df_spots) * 100).sort_values(ascending=False)
print('Null % per column:')
print(null_pct[null_pct > 0])

In [ ]:
# ── Distribution by zone/tariff ───────────────────────────────────────────────
# TODO: update column name once you've seen the actual column names above
# Common column names in this dataset: 'ZONE', 'TARIFF', 'TIPO', 'FREGUESIA'

# Example (adjust column name as needed):
# zone_col = 'ZONE'  # <- check df_spots.columns and update
# print(df_spots[zone_col].value_counts())

In [ ]:
# ── Save raw to disk ──────────────────────────────────────────────────────────
df_spots.to_csv(RAW_DIR + 'emel_spots_raw.csv', index=False)
print(f'✅ Saved to {RAW_DIR}emel_spots_raw.csv')

---
## 2. Dataset 2 — Regulated Parking Areas (Tariff Zones)

**Source:** Lisboa Dados Abertos  
**URL:** https://dadosabertos.cm-lisboa.pt/dataset/areas-reguladas-de-estacionamento-na-via-publica  
**What it contains:** Polygon geometries for each regulated zone — tariff tier (green/yellow/red/brown), schedule, type.

In [ ]:
# ── Fetch GeoJSON ─────────────────────────────────────────────────────────────
ZONES_URL = (
    "https://dadosabertos.cm-lisboa.pt/dataset/"
    "areas-reguladas-de-estacionamento-na-via-publica/"
    "resource/download"
)

# This dataset is typically served as GeoJSON — load with geopandas
# If the URL above doesn't resolve, download manually and load from file:
# gdf_zones = gpd.read_file(RAW_DIR + 'tariff_zones.geojson')

try:
    gdf_zones = gpd.read_file(ZONES_URL)
    print(f'Records: {len(gdf_zones)}')
    print(f'CRS: {gdf_zones.crs}')
    print(f'Columns: {list(gdf_zones.columns)}')
except Exception as e:
    print(f'⚠️  Could not fetch directly: {e}')
    print('→ Download manually from the URL above and load with gpd.read_file()')

In [ ]:
# ── Quick map of tariff zones ─────────────────────────────────────────────────
# Run after gdf_zones is loaded

# ZONE_COLOR_MAP = {
#     'Verde': 'green',
#     'Amarela': 'gold',
#     'Vermelha': 'red',
#     'Castanha': '#8B4513'
# }

# fig, ax = plt.subplots(figsize=(10, 12))
# tariff_col = 'TARIFF'  # <- update once you've seen column names
# gdf_zones.plot(column=tariff_col, ax=ax, legend=True,
#                categorical=True, cmap='tab10', alpha=0.6)
# ax.set_title('Lisbon — Regulated Parking Tariff Zones', fontsize=14)
# ax.axis('off')
# plt.tight_layout()
# plt.savefig(PROCESSED_DIR + 'tariff_zones_map.png', dpi=150)
# plt.show()

print('↑ Uncomment after loading gdf_zones and confirming column names')

---
## 3. Dataset 3 — ZEDL / ZAAC Zone Boundaries

**Source:** Lisboa Dados Abertos  
**URL:** https://dadosabertos.cm-lisboa.pt/dataset/zonas-reguladas-de-estacionamento-na-via-publica  
**What it contains:** Geographic boundaries of ZEDL (limited duration zones) and ZAAC (restricted car access zones) — useful for spatial joins with spot data.

In [ ]:
# ── Fetch zone boundaries ─────────────────────────────────────────────────────
# Download from URL above and load:
# gdf_zone_boundaries = gpd.read_file(RAW_DIR + 'zone_boundaries.geojson')

# Spatial join example (once both datasets are loaded):
# df_spots_geo = gpd.GeoDataFrame(
#     df_spots,
#     geometry=gpd.points_from_xy(df_spots['LONGITUDE'], df_spots['LATITUDE']),
#     crs='EPSG:4326'
# )
# spots_with_zones = gpd.sjoin(df_spots_geo, gdf_zone_boundaries, how='left', predicate='within')

print('↑ Uncomment after loading both datasets')

---
## 4. EMEL Financial Context (Manual — from 2024 Annual Report)

These figures come from EMEL's 2024 *Relatório e Contas*, reported in Jornal Económico (Feb 2026).  
We build a structured reference DataFrame to use in the SQL and Power BI phases.

In [ ]:
# ── EMEL Revenue Reference Table ──────────────────────────────────────────────
emel_financials = pd.DataFrame([
    {'year': 2024, 'stream': 'On-street metered parking', 'revenue_eur': 25_700_000, 'notes': '102,526 spots across 21 parishes'},
    {'year': 2024, 'stream': 'Off-street garages & lots',  'revenue_eur': 14_600_000, 'notes': '36 facilities — estimated from total'},
    {'year': 2024, 'stream': 'Parking fines',              'revenue_eur':  7_200_000, 'notes': '190,952 fines, avg €37.92 — 100% retained from 2024'},
    {'year': 2024, 'stream': 'Clamps, tows & other',       'revenue_eur':  6_571_000, 'notes': '54,506 clamps + 19,219 tows — estimated remainder'},
    {'year': 2023, 'stream': 'Total operating revenue',    'revenue_eur': 47_000_000, 'notes': 'Approximate — pre-fine retention change'},
    {'year': 2024, 'stream': 'Total operating revenue',    'revenue_eur': 54_071_000, 'notes': 'From EMEL Relatório e Contas 2024'},
])

emel_financials['revenue_m'] = emel_financials['revenue_eur'] / 1_000_000

print('EMEL Financial Reference')
print('=' * 60)
emel_financials[['year','stream','revenue_m','notes']].to_string(index=False)

In [ ]:
# ── Revenue breakdown bar chart ───────────────────────────────────────────────
df_2024 = emel_financials[
    (emel_financials['year'] == 2024) & 
    (emel_financials['stream'] != 'Total operating revenue')
].copy()

colors = ['#3b82f6', '#8b5cf6', '#ec4899', '#f59e0b']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(df_2024['stream'], df_2024['revenue_m'], color=colors)
ax.set_xlabel('Revenue (€M)')
ax.set_title('EMEL Revenue by Stream — 2024', fontsize=13, fontweight='bold')

for bar, val in zip(bars, df_2024['revenue_m']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'€{val:.1f}M', va='center', fontsize=10)

ax.set_xlim(0, 32)
plt.tight_layout()
plt.savefig(PROCESSED_DIR + 'emel_revenue_breakdown.png', dpi=150)
plt.show()
print('✅ Chart saved')

In [ ]:
# ── Save financials to processed ──────────────────────────────────────────────
emel_financials.to_csv(PROCESSED_DIR + 'emel_financials.csv', index=False)
print('✅ Saved emel_financials.csv')

---
## 5. Summary & Next Steps

### What we have after Phase 1:
| Output | File | Used in |
|---|---|---|
| Raw spot locations | `data/raw/emel_spots_raw.csv` | Phase 2 — SQL |
| Tariff zone polygons | `data/raw/tariff_zones.geojson` | Phase 2 — SQL + Phase 3 — Power BI |
| Zone boundaries | `data/raw/zone_boundaries.geojson` | Phase 2 — SQL |
| EMEL financial reference | `data/processed/emel_financials.csv` | Phase 2 — SQL + Phase 3 — Power BI |
| Revenue chart | `data/processed/emel_revenue_breakdown.png` | README / portfolio |

### Phase 2 — SQL (next)
- Design relational schema: `spots`, `zones`, `financials`, `parishes`
- Load processed CSVs into MySQL
- Write analytical queries: revenue per zone, spots per parish, tariff distribution

### Open questions to resolve before Phase 2:
- [ ] Confirm actual column names in spot dataset (after running cell 1.3)
- [ ] Decide on CRS — reproject to EPSG:4326 if needed
- [ ] Check whether occupancy data (2020–22) has a joinable key to spot IDs